# Stage 6.1.5 — isolated clean-room Colab checkpoint

This notebook validates the Stage 6.1.5 corrective package from an exact 40-character Git
commit SHA. It creates an isolated virtual environment, installs the declared `.[dev]`
dependencies there, executes the canonical release gate, preserves online historical EPSS
evidence, exports blinded Stage 6.1 packets, optionally processes a completed manual baseline
bundle, preserves any separately supplied controlled-exception adjudication, and emits a hashed
checkpoint archive.

**Before selecting Runtime → Run all:** either paste the exact Git commit SHA into `REF` in the
next cell or leave it blank and paste the SHA when prompted. Do not use `main`, a branch name,
or an abbreviated SHA. If the completed manual bundle is known to require an adjudicated
controlled exception, upload the adjudication JSON separately and set `CONTROLLED_EXCEPTION_JSON`;
do not place governance files inside the canonical six-file manual bundle.**


In [ ]:
from __future__ import annotations

import re
from pathlib import Path

REPO_URL = "https://github.com/richietrap/sbom_to_audit.git"
REF = ""  # Exact 40-character commit SHA. Leave blank to receive an interactive prompt.
MANUAL_RESULTS_ZIP = ""  # Optional uploaded ZIP containing one canonical completed bundle.
CONTROLLED_EXCEPTION_JSON = ""  # Optional separate adjudication JSON; required only on strict RC=1.
RUN_ONLINE_EPSS = True

if not REF.strip():
    REF = input("Paste the exact 40-character Stage 6.1.5 Git commit SHA: ").strip()
if re.fullmatch(r"[0-9a-fA-F]{40}", REF) is None:
    raise ValueError(
        "REF must be an exact 40-character Git commit SHA. Branches, tags, main, master, "
        "and abbreviated SHAs are not accepted by this checkpoint."
    )

manual_results_zip = Path(MANUAL_RESULTS_ZIP).expanduser().resolve() if MANUAL_RESULTS_ZIP else None
controlled_exception_json = (
    Path(CONTROLLED_EXCEPTION_JSON).expanduser().resolve()
    if CONTROLLED_EXCEPTION_JSON
    else None
)
if manual_results_zip is not None and not manual_results_zip.is_file():
    raise FileNotFoundError(f"Manual results ZIP does not exist: {manual_results_zip}")
if controlled_exception_json is not None and not controlled_exception_json.is_file():
    raise FileNotFoundError(
        f"Controlled-exception adjudication does not exist: {controlled_exception_json}"
    )
if controlled_exception_json is not None and manual_results_zip is None:
    raise ValueError("CONTROLLED_EXCEPTION_JSON requires MANUAL_RESULTS_ZIP")

print("Repository:", REPO_URL)
print("Exact commit requested:", REF.lower())
print("Manual result bundle:", manual_results_zip or "not supplied")
print("Controlled-exception adjudication:", controlled_exception_json or "not supplied")
print("Online historical EPSS verification:", RUN_ONLINE_EPSS)


In [ ]:
import json
import os
import platform
import re
import shlex
import shutil
import subprocess
import sys
import time
from pathlib import Path
from collections.abc import Sequence

STAGE = "6.1.5"
PACKAGE_VERSION = "0.6.5"
CHECKPOINT_ID = "STAGE6-1-5-CHECKPOINT-001"
WORKDIR = Path("/content/sbom_to_audit_stage615")
VENV = Path("/content/sbom_to_audit_stage615_venv")
LOG_ROOT = Path("/content/stage615_checkpoint_logs")
RELEASE_REPORT = Path("/content/stage615_release_validation.json")
EPSS_ROOT = Path("/content/stage615_epss_authoritative_evidence")
HISTORICAL_ROOT = Path("/content/stage615_historical_verified")
PACKET_ROOT = Path("/content/stage615_baseline_packets")
IMPORT_ROOT = Path("/content/stage615_imported")
COMPARISON_ROOT = Path("/content/stage615_comparison")
ASSET_ROOT = Path("/content/stage615_paper_assets")
STRICT_VALIDATION_REPORT = Path("/content/stage615_manual_validation.json")
CHECKPOINT_ROOT = Path("/content/stage615_checkpoint_evidence")
ZIP_PATH = Path("/content/stage615_colab_checkpoint_evidence.zip")

for path_to_clear in (
    WORKDIR,
    VENV,
    LOG_ROOT,
    EPSS_ROOT,
    HISTORICAL_ROOT,
    PACKET_ROOT,
    IMPORT_ROOT,
    COMPARISON_ROOT,
    ASSET_ROOT,
    CHECKPOINT_ROOT,
):
    if path_to_clear.exists():
        shutil.rmtree(path_to_clear)
for file_to_clear in (RELEASE_REPORT, STRICT_VALIDATION_REPORT, ZIP_PATH):
    if file_to_clear.exists():
        file_to_clear.unlink()
LOG_ROOT.mkdir(parents=True)


def run_checked(
    name: str,
    command: Sequence[str],
    *,
    cwd: Path | None = None,
    env: dict[str, str] | None = None,
    attempts: int = 1,
    retry_delay_seconds: int = 5,
    accepted_returncodes: Sequence[int] = (0,),
) -> subprocess.CompletedProcess[str]:
    """Run a command, preserve every attempt, and raise outside accepted return codes."""
    if attempts < 1:
        raise ValueError("attempts must be at least one")
    accepted = {int(code) for code in accepted_returncodes}
    if not accepted:
        raise ValueError("accepted_returncodes must not be empty")
    safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", name).strip("_").lower()
    summary_path = LOG_ROOT / f"{safe_name}.summary.json"
    command_list = [str(part) for part in command]
    attempt_records: list[dict[str, object]] = []
    last: subprocess.CompletedProcess[str] | None = None
    for attempt in range(1, attempts + 1):
        completed = subprocess.run(
            command_list,
            cwd=cwd,
            env=env,
            text=True,
            capture_output=True,
            check=False,
        )
        last = completed
        attempt_path = LOG_ROOT / f"{safe_name}.attempt-{attempt:02d}.log"
        attempt_path.write_text(
            "COMMAND: "
            + shlex.join(command_list)
            + f"\nWORKING DIRECTORY: {cwd or Path.cwd()}"
            + f"\nATTEMPT: {attempt}/{attempts}"
            + f"\nRETURN CODE: {completed.returncode}\n"
            + "\n--- STDOUT ---\n"
            + completed.stdout
            + "\n--- STDERR ---\n"
            + completed.stderr,
            encoding="utf-8",
        )
        attempt_records.append(
            {
                "attempt": attempt,
                "returncode": completed.returncode,
                "log": attempt_path.name,
            }
        )
        summary_path.write_text(
            json.dumps(
                {
                    "name": name,
                    "command": command_list,
                    "working_directory": str(cwd or Path.cwd()),
                    "attempts": attempt_records,
                    "final_returncode": completed.returncode,
                    "accepted_returncodes": sorted(accepted),
                },
                indent=2,
                sort_keys=True,
            )
            + "\n",
            encoding="utf-8",
        )
        if completed.returncode in accepted:
            label = "PASS" if completed.returncode == 0 else "ACCEPTED"
            print(f"{label}: {name} (return code {completed.returncode})")
            return completed
        if attempt < attempts:
            print(f"RETRY: {name} failed; waiting {retry_delay_seconds} seconds")
            time.sleep(retry_delay_seconds)
    assert last is not None
    diagnostic = (last.stdout + "\n" + last.stderr).strip()
    if len(diagnostic) > 4000:
        diagnostic = diagnostic[-4000:]
    print(diagnostic)
    raise RuntimeError(
        f"{name} failed with exit code {last.returncode}. Attempt logs: {LOG_ROOT}"
    )


run_checked("clone repository", ["git", "clone", "--no-checkout", REPO_URL, str(WORKDIR)])
run_checked("checkout exact commit", ["git", "checkout", "--detach", REF], cwd=WORKDIR)
commit = run_checked("resolve checked out commit", ["git", "rev-parse", "HEAD"], cwd=WORKDIR)
COMMIT = commit.stdout.strip().lower()
if COMMIT != REF.lower():
    raise RuntimeError(f"Checked out {COMMIT}, but the requested exact commit was {REF.lower()}")
branch = run_checked("verify detached checkout", ["git", "branch", "--show-current"], cwd=WORKDIR)
if branch.stdout.strip():
    raise RuntimeError(f"Checkpoint checkout is attached to mutable branch: {branch.stdout.strip()}")
status = run_checked("verify clean checkout", ["git", "status", "--porcelain"], cwd=WORKDIR)
if status.stdout.strip():
    raise RuntimeError(f"Fresh checkout is unexpectedly dirty:\n{status.stdout}")
os.chdir(WORKDIR)
print("Exact commit checked out:", COMMIT)
print("Kernel Python:", sys.version)
print("Platform:", platform.platform())


In [ ]:
def _write_bootstrap_log(name: str, completed: subprocess.CompletedProcess[str]) -> None:
    (LOG_ROOT / name).write_text(
        "RETURN CODE: "
        + str(completed.returncode)
        + "\n\n--- STDOUT ---\n"
        + completed.stdout
        + "\n--- STDERR ---\n"
        + completed.stderr,
        encoding="utf-8",
    )


stdlib_venv = subprocess.run(
    [sys.executable, "-m", "venv", str(VENV)],
    text=True,
    capture_output=True,
    check=False,
)
_write_bootstrap_log("stdlib_venv_creation.log", stdlib_venv)
VENV_PYTHON = VENV / "bin" / "python"
pip_probe = subprocess.CompletedProcess(args=[], returncode=1, stdout="", stderr="not attempted")
if stdlib_venv.returncode == 0 and VENV_PYTHON.is_file():
    pip_probe = subprocess.run(
        [str(VENV_PYTHON), "-m", "pip", "--version"],
        text=True,
        capture_output=True,
        check=False,
    )
    _write_bootstrap_log("stdlib_venv_pip_probe.log", pip_probe)

if stdlib_venv.returncode != 0 or pip_probe.returncode != 0:
    if VENV.exists():
        shutil.rmtree(VENV)
    print("INFO: stdlib venv was incomplete; using the virtualenv fallback")
    run_checked(
        "install virtualenv fallback",
        [sys.executable, "-m", "pip", "install", "--no-cache-dir", "virtualenv"],
        attempts=3,
    )
    run_checked(
        "create isolated environment with virtualenv",
        [sys.executable, "-m", "virtualenv", str(VENV)],
    )
else:
    print("PASS: create isolated environment with stdlib venv and pip")

VENV_PYTHON = VENV / "bin" / "python"
if not VENV_PYTHON.is_file():
    raise FileNotFoundError(f"Isolated Python was not created: {VENV_PYTHON}")
VENV_ENV = os.environ.copy()
VENV_ENV["PATH"] = f"{VENV / 'bin'}:{VENV_ENV.get('PATH', '')}"
VENV_ENV["VIRTUAL_ENV"] = str(VENV)
VENV_ENV["PYTHONHASHSEED"] = "0"
VENV_ENV["PYTHONNOUSERSITE"] = "1"
VENV_ENV["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
VENV_ENV["PIP_REQUIRE_VIRTUALENV"] = "true"

run_checked(
    "upgrade isolated packaging tools",
    [
        str(VENV_PYTHON),
        "-m",
        "pip",
        "install",
        "--upgrade",
        "pip",
        "setuptools",
        "wheel",
    ],
    env=VENV_ENV,
    attempts=3,
)
run_checked(
    "install package and development dependencies",
    [str(VENV_PYTHON), "-m", "pip", "install", "--no-cache-dir", "-e", ".[dev]"],
    cwd=WORKDIR,
    env=VENV_ENV,
    attempts=3,
)
run_checked(
    "isolated dependency integrity",
    [str(VENV_PYTHON), "-m", "pip", "check"],
    cwd=WORKDIR,
    env=VENV_ENV,
)
run_checked(
    "compile Python sources",
    [str(VENV_PYTHON), "-m", "compileall", "-q", "src", "scripts", "tests"],
    cwd=WORKDIR,
    env=VENV_ENV,
)

isolation_probe_script = (
    "import json, pathlib, site, sys; "
    "print(json.dumps({'prefix': str(pathlib.Path(sys.prefix).resolve()), "
    "'base_prefix': str(pathlib.Path(sys.base_prefix).resolve()), "
    "'user_site_enabled': site.ENABLE_USER_SITE}, sort_keys=True))"
)
isolation_probe = run_checked(
    "verify virtual environment isolation",
    [str(VENV_PYTHON), "-c", isolation_probe_script],
    cwd=WORKDIR,
    env=VENV_ENV,
)
ISOLATION = json.loads(isolation_probe.stdout.strip())
if Path(ISOLATION["prefix"]) != VENV.resolve():
    raise RuntimeError(f"Unexpected isolated sys.prefix: {ISOLATION}")
if ISOLATION["prefix"] == ISOLATION["base_prefix"]:
    raise RuntimeError(f"Interpreter is not running in a virtual environment: {ISOLATION}")
if ISOLATION["user_site_enabled"] is True:
    raise RuntimeError(f"User site-packages remain enabled in the checkpoint environment: {ISOLATION}")

version_probe = run_checked(
    "verify package version",
    [
        str(VENV_PYTHON),
        "-c",
        (
            "import sbom_to_audit; "
            f"assert sbom_to_audit.__version__ == '{PACKAGE_VERSION}', "
            "sbom_to_audit.__version__; print(sbom_to_audit.__version__)"
        ),
    ],
    cwd=WORKDIR,
    env=VENV_ENV,
)
version_script = (
    "import importlib.metadata as m, json; "
    "names=['sbom-to-audit','ruff','mypy','codespell','yamllint','hypothesis',"
    "'pytest','pytest-cov']; "
    "print(json.dumps({name:m.version(name) for name in names}, sort_keys=True))"
)
tool_probe = run_checked(
    "record isolated tool versions",
    [str(VENV_PYTHON), "-c", version_script],
    cwd=WORKDIR,
    env=VENV_ENV,
)
TOOL_VERSIONS = json.loads(tool_probe.stdout.strip())
print("Package version:", version_probe.stdout.strip())
print("Isolation:", json.dumps(ISOLATION, indent=2, sort_keys=True))
print("Isolated tool versions:", json.dumps(TOOL_VERSIONS, indent=2, sort_keys=True))


In [ ]:
run_checked(
    "canonical release gate",
    [str(VENV_PYTHON), "scripts/release_check.py", "--report", str(RELEASE_REPORT)],
    cwd=WORKDIR,
    env=VENV_ENV,
)
release = json.loads(RELEASE_REPORT.read_text(encoding="utf-8"))
if release.get("status") != "PASS":
    raise RuntimeError(f"Canonical release gate did not pass: {release.get('errors')}")
failed_checks = [row for row in release.get("checks", []) if row.get("returncode") != 0]
if failed_checks:
    raise RuntimeError(f"Release report contains failed checks: {failed_checks}")
required_checks = {
    "dependency integrity",
    "compile",
    "ruff lint",
    "ruff format",
    "mypy",
    "codespell",
    "yamllint",
    "historical EPSS offline contract",
    "Stage 6.1 freeze verification",
    "Stage 6.1 control validation",
    "Stage 6.1 blank worksheet validation",
    "Stage 6.1.5 Colab notebook contract",
    "repository validation",
    "tests and coverage",
}
observed_checks = {row["name"] for row in release.get("checks", [])}
missing_checks = sorted(required_checks - observed_checks)
if missing_checks:
    raise RuntimeError(f"Canonical release report omitted required checks: {missing_checks}")
deterministic_hashes = release.get("deterministic_hashes") or {}
if not deterministic_hashes:
    raise RuntimeError("Canonical release report contains no deterministic replay hashes")
for prefix in ("historical_public/", "stage6_baseline/", "stage6_1_packets/"):
    if not any(key.startswith(prefix) for key in deterministic_hashes):
        raise RuntimeError(f"Canonical release report omitted deterministic family: {prefix}")
print("PASS: canonical release report verified")
print("Deterministic hashes:", len(release["deterministic_hashes"]))

EPSS_REPORT = EPSS_ROOT / "historical_epss_verification.json"
if RUN_ONLINE_EPSS:
    run_checked(
        "authoritative online historical EPSS verification",
        [
            str(VENV_PYTHON),
            "scripts/verify_historical_epss.py",
            "--online",
            "--output-dir",
            str(EPSS_ROOT),
            "--report",
            str(EPSS_REPORT),
        ],
        cwd=WORKDIR,
        env=VENV_ENV,
        attempts=3,
        retry_delay_seconds=10,
    )
    epss_verification = json.loads(EPSS_REPORT.read_text(encoding="utf-8"))
    if epss_verification.get("status") != "authoritative_dual_source_verified":
        raise RuntimeError(f"Historical EPSS verification failed: {epss_verification}")
    run_checked(
        "generate online-verified historical replay",
        [
            str(VENV_PYTHON),
            "scripts/run_historical_replay.py",
            "--output-root",
            str(HISTORICAL_ROOT),
            "--epss-verification-report",
            str(EPSS_REPORT),
        ],
        cwd=WORKDIR,
        env=VENV_ENV,
    )
    verified_bundle = (
        HISTORICAL_ROOT
        / "historical_public"
        / "cve_2024_3400_public_bundle.json"
    )
    run_checked(
        "assert online-verified historical replay eligibility",
        [
            str(VENV_PYTHON),
            "scripts/assert_verified_historical_replay.py",
            str(verified_bundle),
        ],
        cwd=WORKDIR,
        env=VENV_ENV,
    )
    EPSS_STATUS = epss_verification["status"]
else:
    epss_verification = {"status": "SKIPPED_BY_CONFIGURATION"}
    EPSS_STATUS = epss_verification["status"]
    print("WARNING: online historical EPSS verification was skipped by configuration")


In [ ]:
import hashlib


def validate_packet_export(packet_root: Path) -> list[dict[str, object]]:
    """Validate packet identity, hashes, paths, and the notebook blinding boundary."""
    packet_registry_path = packet_root / "packet_registry.json"
    packet_registry = json.loads(packet_registry_path.read_text(encoding="utf-8"))
    if packet_registry.get("protocol_version") != "0.2":
        raise RuntimeError(f"Unexpected packet protocol version: {packet_registry}")
    packets = packet_registry.get("packets") or []
    if not isinstance(packets, list) or not packets:
        raise RuntimeError("Packet export produced an empty or invalid registry")

    packet_root_resolved = packet_root.resolve()
    packet_keys: set[tuple[str, str]] = set()
    for raw_row in packets:
        if not isinstance(raw_row, dict):
            raise RuntimeError(f"Packet registry row is not an object: {raw_row!r}")
        row: dict[str, object] = raw_row
        scenario_id = str(row.get("scenario_id") or "")
        event_id = str(row.get("event_id") or "")
        key = (scenario_id, event_id)
        if not all(key) or key in packet_keys:
            raise RuntimeError(f"Invalid or duplicate packet registry key: {key}")
        packet_keys.add(key)

        relative_manifest = Path(str(row.get("manifest_path") or ""))
        if relative_manifest.is_absolute() or ".." in relative_manifest.parts:
            raise RuntimeError(f"Unsafe packet manifest path: {row}")
        manifest_path = (packet_root / relative_manifest).resolve()
        if packet_root_resolved not in manifest_path.parents:
            raise RuntimeError(f"Packet manifest escapes export root: {row}")
        if not manifest_path.is_file():
            raise FileNotFoundError(f"Packet manifest missing: {manifest_path}")
        expected_manifest_hash = str(row.get("manifest_sha256") or "")
        observed_manifest_hash = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
        if observed_manifest_hash != expected_manifest_hash:
            raise RuntimeError(f"Packet manifest hash mismatch: {row}")

        lowered = relative_manifest.as_posix().lower()
        if any(token in lowered for token in ("oracle", "expected_state", "automated_score")):
            raise RuntimeError(f"Blinding boundary violation in packet path: {row}")
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if not isinstance(manifest, dict):
            raise RuntimeError(f"Packet manifest is not an object: {manifest_path}")
        if manifest.get("scenario_id") != scenario_id or manifest.get("event_id") != event_id:
            raise RuntimeError(f"Packet manifest identity mismatch: {row}")
        forbidden_keys = {
            "expected_state",
            "conflict_oracle",
            "automated_score",
            "normalized_claims",
        }
        pending: list[object] = [manifest]
        while pending:
            value = pending.pop()
            if isinstance(value, dict):
                present = forbidden_keys & set(value)
                if present:
                    raise RuntimeError(
                        f"Blinding boundary violation in packet manifest: {sorted(present)}"
                    )
                pending.extend(value.values())
            elif isinstance(value, list):
                pending.extend(value)

        released = manifest.get("released_artifacts") or []
        if not isinstance(released, list):
            raise RuntimeError(f"released_artifacts is not a list: {manifest_path}")
        for artifact in released:
            if not isinstance(artifact, dict):
                raise RuntimeError(f"Released packet artifact is not an object: {artifact!r}")
            filename = str(artifact.get("filename") or "")
            if not filename or Path(filename).name != filename:
                raise RuntimeError(f"Unsafe packet artifact filename: {artifact}")
            artifact_path = manifest_path.parent / filename
            if not artifact_path.is_file():
                raise FileNotFoundError(f"Released packet artifact missing: {artifact_path}")
            observed_artifact_hash = hashlib.sha256(artifact_path.read_bytes()).hexdigest()
            if observed_artifact_hash != artifact.get("sha256"):
                raise RuntimeError(f"Released packet artifact hash mismatch: {artifact}")
    return packets


run_checked(
    "export blinded Stage 6.1 packets",
    [
        str(VENV_PYTHON),
        "scripts/export_stage6_1_baseline_packets.py",
        "--destination",
        str(PACKET_ROOT),
    ],
    cwd=WORKDIR,
    env=VENV_ENV,
)
packets = validate_packet_export(PACKET_ROOT)

oracle_probe = run_checked(
    "count frozen state-oracle events",
    [
        str(VENV_PYTHON),
        "-c",
        (
            "from pathlib import Path; "
            "from sbom_to_audit.baseline.evaluation_oracles import load_state_oracle; "
            "print(len(load_state_oracle(Path('evaluation/oracles/state_oracle_v0.1.yaml'))))"
        ),
    ],
    cwd=WORKDIR,
    env=VENV_ENV,
)
expected_packet_count = int(oracle_probe.stdout.strip())
if len(packets) != expected_packet_count:
    raise RuntimeError(
        f"Packet count {len(packets)} does not match frozen event universe "
        f"{expected_packet_count}"
    )
print("PASS: blinded packet inventory", len(packets))


In [ ]:
import hashlib
import stat
import zipfile
from pathlib import PurePosixPath


MAX_MANUAL_ZIP_MEMBERS = 200
MAX_MANUAL_ZIP_UNCOMPRESSED_BYTES = 100 * 1024 * 1024


def safe_extract_zip(source: Path, destination: Path) -> None:
    """Extract a ZIP after rejecting traversal, collisions, links, and special files."""
    if destination.exists():
        raise FileExistsError(f"Extraction destination already exists: {destination}")
    if not zipfile.is_zipfile(source):
        raise ValueError(f"Manual result is not a valid ZIP archive: {source}")

    with zipfile.ZipFile(source) as archive:
        members = archive.infolist()
        if not members:
            raise ValueError("Manual ZIP is empty")
        if len(members) > MAX_MANUAL_ZIP_MEMBERS:
            raise ValueError(f"Manual ZIP contains too many entries: {len(members)}")
        total_size = sum(member.file_size for member in members)
        if total_size > MAX_MANUAL_ZIP_UNCOMPRESSED_BYTES:
            raise ValueError(
                "Manual ZIP expands beyond the 100 MiB safety limit: "
                f"{total_size}"
            )

        validated: list[tuple[zipfile.ZipInfo, PurePosixPath, bool]] = []
        normalized_paths: set[str] = set()
        file_paths: set[PurePosixPath] = set()
        directory_paths: set[PurePosixPath] = set()
        for member in members:
            name = member.filename
            is_directory = member.is_dir()
            canonical_name = name[:-1] if is_directory and name.endswith("/") else name
            if member.flag_bits & 0x1:
                raise ValueError(f"Encrypted ZIP members are not accepted: {name}")
            if "\x00" in canonical_name:
                raise ValueError("Manual ZIP contains a NUL byte in a member path")
            if "\\" in canonical_name:
                raise ValueError(f"Backslash ZIP paths are not accepted: {name}")
            if re.match(r"^[A-Za-z]:", canonical_name):
                raise ValueError(f"Drive-qualified ZIP path is not accepted: {name}")
            raw_parts = canonical_name.split("/")
            if not canonical_name or any(part in {"", ".", ".."} for part in raw_parts):
                raise ValueError(f"Unsafe or non-canonical ZIP path: {name}")
            member_path = PurePosixPath(*raw_parts)
            if member_path.is_absolute():
                raise ValueError(f"Unsafe ZIP path: {name}")
            normalized = member_path.as_posix()
            if normalized in normalized_paths:
                raise ValueError(f"Manual ZIP contains duplicate normalized paths: {normalized}")
            normalized_paths.add(normalized)

            mode = member.external_attr >> 16
            file_type = stat.S_IFMT(mode)
            if file_type == stat.S_IFLNK:
                raise ValueError(f"Symbolic links are not accepted in the manual ZIP: {name}")
            if file_type not in (0, stat.S_IFREG, stat.S_IFDIR):
                raise ValueError(f"Special files are not accepted in the manual ZIP: {name}")
            if file_type == stat.S_IFDIR and not is_directory:
                raise ValueError(f"Directory metadata conflicts with member path: {name}")
            if is_directory and member.file_size != 0:
                raise ValueError(f"Directory ZIP member contains file data: {name}")

            if is_directory:
                directory_paths.add(member_path)
            else:
                file_paths.add(member_path)
            validated.append((member, member_path, is_directory))

        for file_path in file_paths:
            if file_path in directory_paths:
                raise ValueError(f"ZIP path is both a file and directory: {file_path}")
            if any(parent in file_paths for parent in file_path.parents):
                raise ValueError(f"ZIP file path is nested below another file: {file_path}")
        for directory_path in directory_paths:
            if any(parent in file_paths for parent in directory_path.parents):
                raise ValueError(
                    f"ZIP directory path is nested below a file: {directory_path}"
                )

        try:
            corrupt_member = archive.testzip()
        except (zipfile.BadZipFile, EOFError, OSError, RuntimeError) as exc:
            raise ValueError(f"Manual ZIP integrity validation failed: {exc}") from exc
        if corrupt_member is not None:
            raise ValueError(f"Manual ZIP contains a corrupt member: {corrupt_member}")

        destination.mkdir(parents=True, exist_ok=False)
        root = destination.resolve()
        extracted_bytes = 0
        for member, member_path, is_directory in validated:
            target = root.joinpath(*member_path.parts)
            resolved_target = target.resolve()
            if resolved_target != root and root not in resolved_target.parents:
                raise ValueError(f"ZIP member escapes extraction root: {member.filename}")
            if is_directory:
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member, "r") as source_stream, target.open("xb") as target_stream:
                while chunk := source_stream.read(1024 * 1024):
                    extracted_bytes += len(chunk)
                    if extracted_bytes > MAX_MANUAL_ZIP_UNCOMPRESSED_BYTES:
                        raise ValueError("Manual ZIP exceeded its extraction-size safety limit")
                    target_stream.write(chunk)


MANUAL_STATUS = "NOT_SUPPLIED"
MANUAL_ZIP_SHA256 = None
MANUAL_STRICT_VALID = None
MANUAL_STRICT_ERROR_COUNT = None
CONTROLLED_EXCEPTION_STATUS = "NOT_SUPPLIED"
CONTROLLED_EXCEPTION_SHA256 = None
if manual_results_zip is not None:
    manual_zip_digest_before = hashlib.sha256(manual_results_zip.read_bytes()).hexdigest()
    controlled_exception_digest_before = (
        hashlib.sha256(controlled_exception_json.read_bytes()).hexdigest()
        if controlled_exception_json is not None
        else None
    )
    BUNDLE_ROOT = Path("/content/stage615_manual_results")
    if BUNDLE_ROOT.exists():
        shutil.rmtree(BUNDLE_ROOT)
    safe_extract_zip(manual_results_zip, BUNDLE_ROOT)
    candidates = sorted(BUNDLE_ROOT.rglob("declaration.yaml"))
    if len(candidates) != 1:
        raise ValueError(
            "Manual result ZIP must contain exactly one canonical bundle with declaration.yaml; "
            f"found {len(candidates)}"
        )
    bundle = candidates[0].parent
    unexpected_files = sorted(
        path
        for path in BUNDLE_ROOT.rglob("*")
        if path.is_file() and path != candidates[0] and bundle not in path.parents
    )
    if unexpected_files:
        raise ValueError(
            "Manual result ZIP contains files outside the canonical bundle: "
            f"{[str(path.relative_to(BUNDLE_ROOT)) for path in unexpected_files]}"
        )
    strict_validation = run_checked(
        "record strict completed manual baseline validation",
        [
            str(VENV_PYTHON),
            "scripts/validate_manual_baseline_worksheet.py",
            str(bundle),
            "--require-complete",
            "--json-output",
            str(STRICT_VALIDATION_REPORT),
        ],
        cwd=WORKDIR,
        env=VENV_ENV,
        accepted_returncodes=(0, 1),
    )
    strict_report = json.loads(STRICT_VALIDATION_REPORT.read_text(encoding="utf-8"))
    MANUAL_STRICT_VALID = strict_report.get("valid")
    MANUAL_STRICT_ERROR_COUNT = len(strict_report.get("errors") or [])
    if strict_validation.returncode == 0:
        if MANUAL_STRICT_VALID is not True or MANUAL_STRICT_ERROR_COUNT != 0:
            raise RuntimeError(f"Strict manual validation result is inconsistent: {strict_report}")
        if controlled_exception_json is not None:
            raise RuntimeError(
                "Controlled-exception adjudication was supplied but strict manual validation passed"
            )
        import_exception_args: list[str] = []
        CONTROLLED_EXCEPTION_STATUS = "NOT_REQUIRED"
    else:
        if MANUAL_STRICT_VALID is not False:
            raise RuntimeError(f"Strict manual validation RC=1 without valid:false: {strict_report}")
        if MANUAL_STRICT_ERROR_COUNT != 1:
            raise RuntimeError(
                "Controlled admission requires exactly one registered strict validation error"
            )
        if controlled_exception_json is None:
            raise RuntimeError(
                "Strict manual validation failed; CONTROLLED_EXCEPTION_JSON is required"
            )
        import_exception_args = [
            "--controlled-exception",
            str(controlled_exception_json),
        ]

    run_checked(
        "import and preserve manual baseline bundle",
        [
            str(VENV_PYTHON),
            "scripts/import_manual_baseline_results.py",
            str(bundle),
            "--destination",
            str(IMPORT_ROOT),
            *import_exception_args,
        ],
        cwd=WORKDIR,
        env=VENV_ENV,
    )
    imported_validation_path = IMPORT_ROOT / "integrity" / "import_validation_report.json"
    imported_validation = json.loads(imported_validation_path.read_text(encoding="utf-8"))
    if imported_validation != strict_report:
        raise RuntimeError("Imported strict validation report differs from the recorded strict result")

    admission_path = IMPORT_ROOT / "integrity" / "controlled_exception_admission.json"
    if strict_validation.returncode == 1:
        if not admission_path.is_file():
            raise FileNotFoundError(f"Controlled-exception admission missing: {admission_path}")
        admission = json.loads(admission_path.read_text(encoding="utf-8"))
        if admission.get("admitted") is not True:
            raise RuntimeError(f"Controlled exception was not admitted: {admission}")
        if admission.get("strict_validation_valid") is not False:
            raise RuntimeError(f"Admission rewrote strict validation state: {admission}")
        if admission.get("strict_validation_errors") != strict_report.get("errors"):
            raise RuntimeError(f"Admission errors differ from strict validation: {admission}")
        if admission.get("adjudication_sha256") != controlled_exception_digest_before:
            raise RuntimeError(f"Admission adjudication hash mismatch: {admission}")
        CONTROLLED_EXCEPTION_STATUS = "ADMITTED"
        CONTROLLED_EXCEPTION_SHA256 = controlled_exception_digest_before
    elif admission_path.exists():
        raise RuntimeError("Unexpected controlled-exception admission for a strict-valid bundle")

    normalized = IMPORT_ROOT / "normalized" / "stage6_1_manual_baseline_normalized.json"
    if not normalized.is_file():
        raise FileNotFoundError(f"Normalized manual result missing: {normalized}")
    run_checked(
        "run Stage 6.1 matched comparison",
        [
            str(VENV_PYTHON),
            "scripts/run_stage6_1_comparison.py",
            str(normalized),
            "--destination",
            str(COMPARISON_ROOT),
        ],
        cwd=WORKDIR,
        env=VENV_ENV,
    )
    comparison_report_path = COMPARISON_ROOT / "comparison" / "stage6_1_comparison_report.json"
    run_checked(
        "validate Stage 6.1 candidate comparison",
        [
            str(VENV_PYTHON),
            "scripts/validate_stage6_1_evaluation.py",
            "--comparison-report",
            str(comparison_report_path),
        ],
        cwd=WORKDIR,
        env=VENV_ENV,
    )
    run_checked(
        "build Stage 6.1 candidate paper assets",
        [
            str(VENV_PYTHON),
            "scripts/build_stage6_1_paper_assets.py",
            str(COMPARISON_ROOT / "comparison"),
            "--destination",
            str(ASSET_ROOT),
        ],
        cwd=WORKDIR,
        env=VENV_ENV,
    )
    comparison_report = json.loads(comparison_report_path.read_text(encoding="utf-8"))
    if comparison_report.get("manuscript_eligible") is not False:
        raise RuntimeError("Stage 6.1 candidate comparison must remain manuscript-ineligible")
    if comparison_report.get("evaluation_status") != "CANDIDATE_NOT_FROZEN":
        raise RuntimeError(f"Unexpected comparison status: {comparison_report}")
    MANUAL_ZIP_SHA256 = hashlib.sha256(manual_results_zip.read_bytes()).hexdigest()
    if MANUAL_ZIP_SHA256 != manual_zip_digest_before:
        raise RuntimeError("Manual result ZIP changed while the checkpoint was processing it")
    if controlled_exception_json is not None:
        controlled_exception_digest_after = hashlib.sha256(
            controlled_exception_json.read_bytes()
        ).hexdigest()
        if controlled_exception_digest_after != controlled_exception_digest_before:
            raise RuntimeError(
                "Controlled-exception adjudication changed while the checkpoint was processing it"
            )
    MANUAL_STATUS = (
        "CONTROLLED_EXCEPTION_ADMITTED_IMPORTED_AND_COMPARED"
        if CONTROLLED_EXCEPTION_STATUS == "ADMITTED"
        else "VALIDATED_IMPORTED_AND_COMPARED"
    )
    print("PASS: optional manual baseline path completed")
else:
    print("PASS: pre-execution checkpoint; no manual result bundle supplied")


In [ ]:
import hashlib
import json
import platform
import shutil
import sys
import zipfile
from datetime import datetime, timezone

CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=False)
controls_root = CHECKPOINT_ROOT / "frozen_controls"
controls_root.mkdir()
for relative in (
    "evaluation/freeze/stage6_1_protocol_freeze.json",
    "evaluation/baseline_protocol_v0.2.yaml",
    "evaluation/oracles/state_oracle_v0.1.yaml",
    "evaluation/oracles/conflict_oracle_v0.1.yaml",
    "evaluation/oracles/clock_opportunity_oracle_v0.1.yaml",
    "evaluation/mappings/common_field_set_v0.1.yaml",
    "evaluation/mappings/traceability_element_mapping_v0.1.yaml",
    "evaluation/mappings/source_access_accounting_v0.1.yaml",
    "evaluation/mappings/equivalent_record_bundle_v0.1.yaml",
):
    source = WORKDIR / relative
    target = controls_root / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)

isolated_python_version = run_checked(
    "record isolated Python version",
    [str(VENV_PYTHON), "--version"],
    cwd=WORKDIR,
    env=VENV_ENV,
).stdout.strip()

shutil.copy2(RELEASE_REPORT, CHECKPOINT_ROOT / RELEASE_REPORT.name)
shutil.copytree(LOG_ROOT, CHECKPOINT_ROOT / "command_logs")
shutil.copytree(PACKET_ROOT, CHECKPOINT_ROOT / "baseline_packets")
if RUN_ONLINE_EPSS:
    shutil.copytree(EPSS_ROOT, CHECKPOINT_ROOT / "historical_epss_evidence")
    shutil.copytree(HISTORICAL_ROOT, CHECKPOINT_ROOT / "verified_historical_replay")
if manual_results_zip is not None:
    shutil.copy2(manual_results_zip, CHECKPOINT_ROOT / "manual_results_original.zip")
    shutil.copy2(STRICT_VALIDATION_REPORT, CHECKPOINT_ROOT / "manual_validation_report.json")
    if controlled_exception_json is not None:
        shutil.copy2(
            controlled_exception_json,
            CHECKPOINT_ROOT / "controlled_exception_adjudication.json",
        )
    shutil.copytree(IMPORT_ROOT, CHECKPOINT_ROOT / "manual_import")
    shutil.copytree(COMPARISON_ROOT, CHECKPOINT_ROOT / "comparison")
    shutil.copytree(ASSET_ROOT, CHECKPOINT_ROOT / "paper_assets")

checkpoint_status = "PASS" if RUN_ONLINE_EPSS else "PARTIAL_ONLINE_EPSS_SKIPPED"
environment = {
    "checkpoint_id": CHECKPOINT_ID,
    "checkpoint_status": checkpoint_status,
    "stage": STAGE,
    "package_version": PACKAGE_VERSION,
    "repository": REPO_URL,
    "requested_ref": REF.lower(),
    "git_commit": COMMIT,
    "generated_at": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    "kernel_python": sys.version,
    "isolated_python": isolated_python_version,
    "isolation": ISOLATION,
    "platform": platform.platform(),
    "tool_versions": TOOL_VERSIONS,
    "release_status": release["status"],
    "release_check_count": len(release.get("checks", [])),
    "deterministic_hash_count": len(release.get("deterministic_hashes", {})),
    "historical_epss_status": EPSS_STATUS,
    "manual_results_status": MANUAL_STATUS,
    "manual_results_zip_sha256": MANUAL_ZIP_SHA256,
    "manual_strict_validation_valid": MANUAL_STRICT_VALID,
    "manual_strict_validation_error_count": MANUAL_STRICT_ERROR_COUNT,
    "controlled_exception_status": CONTROLLED_EXCEPTION_STATUS,
    "controlled_exception_sha256": CONTROLLED_EXCEPTION_SHA256,
}
(CHECKPOINT_ROOT / "checkpoint_environment.json").write_text(
    json.dumps(environment, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

checksums: dict[str, str] = {}
for evidence_path in sorted(CHECKPOINT_ROOT.rglob("*")):
    if evidence_path.is_file():
        relative = evidence_path.relative_to(CHECKPOINT_ROOT).as_posix()
        checksums[relative] = hashlib.sha256(evidence_path.read_bytes()).hexdigest()
checksum_payload = {
    "algorithm": "sha256",
    "scope": "all checkpoint evidence files except this inventory and the outer ZIP",
    "files": checksums,
}
(CHECKPOINT_ROOT / "evidence_checksums.json").write_text(
    json.dumps(checksum_payload, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
for relative, expected_digest in checksums.items():
    observed_digest = hashlib.sha256((CHECKPOINT_ROOT / relative).read_bytes()).hexdigest()
    if observed_digest != expected_digest:
        raise RuntimeError(f"Evidence changed while checksums were being generated: {relative}")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for evidence_path in sorted(CHECKPOINT_ROOT.rglob("*")):
        if evidence_path.is_file():
            archive.write(evidence_path, evidence_path.relative_to(CHECKPOINT_ROOT))
with zipfile.ZipFile(ZIP_PATH) as archive:
    corrupt_member = archive.testzip()
    if corrupt_member is not None:
        raise RuntimeError(f"Checkpoint ZIP integrity failure: {corrupt_member}")
    archive_names = archive.namelist()
    if len(archive_names) != len(set(archive_names)):
        raise RuntimeError("Checkpoint ZIP contains duplicate member paths")
    required_members = {
        "checkpoint_environment.json",
        "evidence_checksums.json",
        RELEASE_REPORT.name,
        "baseline_packets/packet_registry.json",
        "frozen_controls/evaluation/freeze/stage6_1_protocol_freeze.json",
    }
    if RUN_ONLINE_EPSS:
        required_members.update(
            {
                "historical_epss_evidence/historical_epss_verification.json",
                "verified_historical_replay/historical_public/"
                "cve_2024_3400_public_bundle.json",
            }
        )
    if manual_results_zip is not None:
        required_members.update(
            {
                "manual_results_original.zip",
                "manual_validation_report.json",
                "manual_import/normalized/stage6_1_manual_baseline_normalized.json",
                "comparison/comparison/stage6_1_comparison_report.json",
            }
        )
        if CONTROLLED_EXCEPTION_STATUS == "ADMITTED":
            required_members.update(
                {
                    "controlled_exception_adjudication.json",
                    "manual_import/integrity/controlled_exception_admission.json",
                }
            )
    missing_members = sorted(required_members - set(archive_names))
    if missing_members:
        raise RuntimeError(f"Checkpoint ZIP omitted required evidence: {missing_members}")
    archived_inventory = json.loads(archive.read("evidence_checksums.json"))
    for relative, expected_digest in archived_inventory["files"].items():
        if relative not in archive_names:
            raise RuntimeError(f"Checksum inventory references a missing archive member: {relative}")
        observed_digest = hashlib.sha256(archive.read(relative)).hexdigest()
        if observed_digest != expected_digest:
            raise RuntimeError(f"Archived evidence checksum mismatch: {relative}")

zip_sha256 = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
print("PASS: checkpoint evidence archive verified")
print("Checkpoint ZIP:", ZIP_PATH)
print("Exact Git commit:", COMMIT)
print("Checkpoint status:", checkpoint_status)
print("SHA-256:", zip_sha256)

try:
    from IPython.display import FileLink, display

    display(FileLink(str(ZIP_PATH)))
except ImportError:
    pass


## Acceptance boundary

A successful notebook run must finish with `Checkpoint status: PASS`, print the exact Git
commit, and produce `stage615_colab_checkpoint_evidence.zip` with its SHA-256. Preserve the ZIP
unchanged. If online EPSS verification is deliberately disabled, the checkpoint is partial and
must not be treated as the final clean-room acceptance record.
